# SOCOFing fingerprint vault — train and test

Juels–Wattenberg fuzzy commitment, Hadamard-128 error correction, and a Schnorr 3-pass zero-knowledge proof.

**Before you run this notebook**

1. In the Kaggle sidebar, choose **Add Input**.
2. Search for `socofing` and add the dataset by ruizgara.
3. Use **Run all**. Internet is not required after the dataset is attached.

The split is by **subject**, 80% train and 20% test, seed 42. A genuine trial is the real scan of a finger against an altered scan of **that same finger** (central rotation, obliteration, or Z-cut). An impostor trial is a different subject. The vault stores only helper data `W`, hash `H`, and public key `Y`.

When it finishes, download from the output:

- `evaluation_results.json`
- `kaggle_train_summary.png`
- `kaggle_test_summary.png`


In [ ]:
import os, re, json, math, time, hashlib, base64
from collections import defaultdict

import cv2
import numpy as np
import scipy.linalg
import matplotlib
try:
    get_ipython()
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

# How the benchmark is scored. Change these if you want a heavier run.
TRAIN_FRACTION = 0.80
SPLIT_SEED = 42
DIFFICULTY = "easy"          # "easy" or "all"
MAX_GENUINE_PER_FINGER = 3   # one CR, one Obl, one Zcut
IMPOSTORS_PER_FINGER = 8

print("OpenCV", cv2.__version__, "| NumPy", np.__version__)
print("Ready.")


In [ ]:
# ---------------------------------------------------------------------------
# Find every SOCOFing image and group it by finger
# ---------------------------------------------------------------------------
FINGER_RE = re.compile(
    r"^(?P<sid>\d+)__(?P<gender>[MFmf])_(?P<hand>Left|Right)_(?P<finger>[A-Za-z]+)_finger(?P<suffix>.*)\.(?:bmp|tif|png|jpg|jpeg)$",
    re.IGNORECASE,
)

def difficulty_of(path):
    parts = [p.lower() for p in re.split(r"[\\/]", path)]
    blob = " ".join(parts)
    if "altered-hard" in blob or "hard" in parts:
        return "hard"
    if "altered-medium" in blob or "medium" in parts:
        return "medium"
    if "altered" in blob or "easy" in parts:
        return "easy"
    return "real"

def alteration_of(suffix, path):
    if difficulty_of(path) == "real":
        return None
    text = suffix.lower()
    if "zcut" in text:
        return "Zcut"
    if "obl" in text:
        return "Obl"
    if "cr" in text or "rot" in text:
        return "CR"
    return "Unknown"

def find_dataset_root():
    forced = os.environ.get("SOCOFING_DIR")
    if forced and os.path.isdir(forced):
        print("Using SOCOFING_DIR =", forced)
        return forced
    roots = []
    if os.path.isdir("/kaggle/input"):
        roots.append("/kaggle/input")
    roots.extend(["data/kaggle_socofing", "SOCOFing", "."])
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            if any(name.lower().endswith(".bmp") for name in files):
                print("Found fingerprint files under", root)
                return root
    return None

DATASET_DIR = find_dataset_root()
fingers = {}
bmp_count = 0

if DATASET_DIR:
    for dirpath, _, files in os.walk(DATASET_DIR):
        for name in files:
            if not name.lower().endswith((".bmp", ".tif", ".png", ".jpg", ".jpeg")):
                continue
            match = FINGER_RE.match(name)
            if not match:
                continue
            bmp_count += 1
            sid = int(match.group("sid"))
            finger_key = "{:04d}_{}_{}_{}".format(
                sid,
                match.group("gender").upper(),
                match.group("hand").capitalize(),
                match.group("finger").lower(),
            )
            full = os.path.join(dirpath, name)
            slot = fingers.setdefault(finger_key, {"sid": sid, "real": None, "altered": []})
            kind = difficulty_of(full)
            if kind == "real":
                if slot["real"] is None:
                    slot["real"] = full
            else:
                slot["altered"].append({
                    "path": full,
                    "difficulty": kind,
                    "alt": alteration_of(match.group("suffix"), full),
                })

subjects = sorted({slot["sid"] for slot in fingers.values() if slot["real"]})
print("Images matched to the SOCOFing name pattern:", bmp_count)
print("Subjects with a real scan:", len(subjects))
print("Fingers with a real scan:", sum(1 for slot in fingers.values() if slot["real"]))

if not subjects:
    print("=" * 70)
    print("SOCOFing was not found.")
    print("Add the ruizgara/socofing dataset with the Input '+' button, then run this cell again.")
    print("=" * 70)
    raise FileNotFoundError("SOCOFing dataset is not attached.")


In [ ]:
# ---------------------------------------------------------------------------
# 1024-bit Gabor template, Hadamard-128, fuzzy commitment, Schnorr ZKP
# ---------------------------------------------------------------------------
def build_gabor_bank(num_angles=8, frequency=0.10, ksize=21):
    bank = []
    lambd = 1.0 / frequency
    for i in range(num_angles):
        theta = i * np.pi / num_angles
        kernel = cv2.getGaborKernel((ksize, ksize), 4.0, theta, lambd, 0.5, 0, ktype=cv2.CV_32F)
        kernel = kernel - kernel.mean()
        bank.append(kernel)
    return bank

GABOR_BANK = build_gabor_bank()

def extract_binary_template(image, center=None, shift=(0, 0), num_bands=8, num_sectors=16, r_in=15, r_out=120):
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    height, width = image.shape
    min_dim = min(height, width)
    if r_out > int(min_dim * 0.48):
        r_in = max(4, int(min_dim * 0.06))
        r_out = max(r_in + 12, int(min_dim * 0.46))
    if center is None:
        cx, cy = width // 2, height // 2
    else:
        cx, cy = center
    cx += shift[0]
    cy += shift[1]
    img_f = image.astype(np.float32)
    mean, std = img_f.mean(), img_f.std() + 1e-6
    norm = np.clip(((img_f - mean) / std) * 50 + 128, 0, 255).astype(np.uint8)
    filtered = [cv2.filter2D(norm, cv2.CV_32F, kernel) for kernel in GABOR_BANK]
    y, x = np.ogrid[:height, :width]
    dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    angle = np.arctan2(y - cy, x - cx) % (2.0 * np.pi)
    band_width = (r_out - r_in) / float(num_bands)
    sector_width = (2.0 * np.pi) / float(num_sectors)
    sec_idx = np.floor(angle / sector_width).astype(np.int32)
    band_idx = np.floor((dist - r_in) / band_width).astype(np.int32)
    valid = (dist >= r_in) & (dist < r_out) & (sec_idx >= 0) & (sec_idx < num_sectors) & (band_idx >= 0) & (band_idx < num_bands)
    n_regions = num_bands * num_sectors
    region_id = np.full((height, width), -1, dtype=np.int32)
    region_id[valid] = band_idx[valid] * num_sectors + sec_idx[valid]
    reg_flat = region_id.ravel()
    valid_flat = reg_flat >= 0
    reg_valid = reg_flat[valid_flat]
    counts = np.maximum(np.bincount(reg_valid, minlength=n_regions).astype(np.float32), 1.0)
    features = np.zeros((n_regions, len(filtered)), dtype=np.float32)
    for k in range(len(filtered)):
        values = filtered[k].ravel()[valid_flat]
        means = np.bincount(reg_valid, weights=values, minlength=n_regions) / counts
        deviation = np.abs(values - means[reg_valid])
        features[:, k] = np.bincount(reg_valid, weights=deviation, minlength=n_regions) / counts
    flat = features.ravel()
    return (flat > np.median(flat)).astype(np.uint8)

def sector_rotations(template, num_bands=8, num_sectors=16, num_filters=8):
    flat = np.asarray(template, dtype=np.uint8).reshape(-1)
    cube = flat.reshape(num_bands, num_sectors, num_filters)
    shifted = np.empty((num_sectors, flat.size), dtype=np.uint8)
    for step in range(num_sectors):
        shifted[step] = np.roll(cube, step, axis=1).ravel()
    return shifted

HADAMARD_128 = scipy.linalg.hadamard(128).astype(np.float32)

def encode_byte_hadamard(byte_val):
    vector = HADAMARD_128[byte_val % 128].copy()
    if byte_val >= 128:
        vector = -vector
    return ((1.0 - vector) / 2.0).astype(np.uint8)

def decode_byte_hadamard(received_128):
    mapped = 1.0 - 2.0 * received_128.astype(np.float32)
    corr = np.dot(HADAMARD_128, mapped)
    max_idx = int(np.argmax(np.abs(corr)))
    peak = corr[max_idx]
    if peak >= 0:
        return max_idx, float(abs(peak))
    return max_idx + 128, float(abs(peak))

def ecc_encode(secret_bytes):
    return np.concatenate([encode_byte_hadamard(b) for b in secret_bytes]).astype(np.uint8)

def ecc_decode(codeword_bits):
    recovered, peaks = [], []
    for block in codeword_bits.reshape(8, 128):
        byte_val, peak = decode_byte_hadamard(block)
        recovered.append(byte_val)
        peaks.append(peak)
    return bytes(recovered), peaks

INTERLEAVE_INDEX = np.random.RandomState(20241120).permutation(1024).astype(np.int32)

def interleave_bits(bits):
    return np.asarray(bits, dtype=np.uint8).reshape(-1)[INTERLEAVE_INDEX]

# RFC 3526 1024-bit MODP group. q = (p - 1) / 2 and g = 2, same as the web app.
P = int(
    "FFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD1"
    "29024E088A67CC74020BBEA63B139B22514A08798E3404DD"
    "EF9519B3CD3A431B302B0A6DF25F14374FE1356D6D51C245"
    "E485B576625E7EC6F44C42E9A637ED6B0BFF5CB6F406B7ED"
    "EE386BFB5A899FA5AE9F24117C4B1FE649286651ECE45B3D"
    "C2007CB8A163BF0598DA48361C55D39A69163FA8FD24CF5F"
    "83655D23DCA3AD961C62F356208552BB9ED529077096966D"
    "670C354E4ABC9804F1746C08CA18217C32905E462E36CE3B"
    "E39E772C180E86039B2783A2EC07A28FB5C55DF06F4C52C9"
    "DE2BCBF6955817183995497CEA956AE515D2261898FA0510"
    "15728E5A8AACAA68FFFFFFFFFFFFFFFF", 16)
Q = (P - 1) // 2
G = 2

def fuzzy_commit(template_b):
    secret = os.urandom(8)
    private = int.from_bytes(secret, "big") % Q or 1
    helper = interleave_bits(template_b) ^ ecc_encode(secret)
    commit = hashlib.sha256(secret).hexdigest()
    public = pow(G, private, P)
    return helper.astype(np.uint8), commit, public, secret

def fuzzy_recover(template_q, helper, commit):
    noisy = interleave_bits(template_q) ^ helper
    recovered, peaks = ecc_decode(noisy)
    ok = hashlib.sha256(recovered).hexdigest() == commit
    flips = int(sum(max(0, int(round((128.0 - peak) / 2.0))) for peak in peaks))
    return ok, (recovered if ok else None), flips

def schnorr_verify(secret, public_key):
    private = int.from_bytes(secret, "big") % Q or 1
    nonce = int.from_bytes(os.urandom(32), "big") % (Q - 1) + 1
    commitment = pow(G, nonce, P)
    challenge = int.from_bytes(os.urandom(20), "big") % (Q - 1) + 1
    response = (nonce + challenge * private) % Q
    lhs = pow(G, response, P)
    rhs = (commitment * pow(public_key, challenge, P)) % P
    return lhs == rhs

# Self-test. The notebook stops here if the cryptosystem is wrong.
_secret = b"TEST_SEC"
_codeword = ecc_encode(_secret)
assert ecc_decode(_codeword)[0] == _secret
_noise = np.zeros(1024, dtype=np.uint8)
_noise[:20] = 1
assert ecc_decode(_codeword ^ _noise)[0] == _secret
_template = np.random.RandomState(0).randint(0, 2, 1024).astype(np.uint8)
_w, _h, _y, _c = fuzzy_commit(_template)
_flip = np.zeros(1024, dtype=np.uint8)
_flip[:80] = 1
assert fuzzy_recover(_template ^ _flip, _w, _h)[0] is True
assert fuzzy_recover(1 - _template, _w, _h)[0] is False
assert schnorr_verify(_c, _y) is True
print("Crypto self-test passed (ECC, fuzzy commitment, Schnorr).")


In [ ]:
# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def best_hamming(enrolled, query):
    shifts = sector_rotations(query)
    return float(np.min(np.mean(shifts != enrolled, axis=1)))

def crypto_decision(query, helper, commit, public_key):
    for candidate in sector_rotations(query):
        matched, recovered, _flips = fuzzy_recover(candidate, helper, commit)
        if matched:
            return True, schnorr_verify(recovered, public_key)
    return False, False

def roc_and_eer(genuine, impostor):
    genuine = np.asarray(genuine, dtype=np.float64)
    impostor = np.asarray(impostor, dtype=np.float64)
    thresholds = np.linspace(0.0, 1.0, 501)
    far = np.array([np.mean(impostor <= t) for t in thresholds])
    frr = np.array([np.mean(genuine > t) for t in thresholds])
    gar = 1.0 - frr
    eer_at = int(np.argmin(np.abs(far - frr)))
    eer = float((far[eer_at] + frr[eer_at]) / 2.0)
    order = np.argsort(far)
    trap = getattr(np, "trapezoid", None) or getattr(np, "trapz")
    auc = float(trap(gar[order], far[order]))
    step = 10
    return {
        "eer": eer,
        "eer_percentage": round(eer * 100.0, 3),
        "eer_threshold": round(float(thresholds[eer_at]), 4),
        "auc": round(auc, 4),
        "mean_genuine_hd": round(float(genuine.mean()), 4),
        "std_genuine_hd": round(float(genuine.std()), 4),
        "min_genuine_hd": round(float(genuine.min()), 4),
        "max_genuine_hd": round(float(genuine.max()), 4),
        "mean_impostor_hd": round(float(impostor.mean()), 4),
        "std_impostor_hd": round(float(impostor.std()), 4),
        "min_impostor_hd": round(float(impostor.min()), 4),
        "max_impostor_hd": round(float(impostor.max()), 4),
        "separation_margin": round(float(impostor.mean() - genuine.mean()), 4),
        "curve": {
            "thresholds": [round(float(v), 4) for v in thresholds[::step]],
            "far": [round(float(v), 6) for v in far[::step]],
            "frr": [round(float(v), 6) for v in frr[::step]],
            "gar": [round(float(v), 6) for v in gar[::step]],
        },
    }

def save_summary_plot(genuine, impostor, metrics, title, path):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    axes[0].hist(genuine, bins=30, density=True, alpha=0.7, color="#10b981", label="Genuine")
    axes[0].hist(impostor, bins=30, density=True, alpha=0.7, color="#ef4444", label="Impostor")
    axes[0].axvline(metrics["eer_threshold"], color="#6366f1", linestyle="--", label="EER threshold")
    axes[0].set_title("Hamming distance")
    axes[0].legend()
    curve = metrics["curve"]
    axes[1].plot(curve["thresholds"], [v * 100 for v in curve["far"]], color="#ef4444", label="FAR")
    axes[1].plot(curve["thresholds"], [v * 100 for v in curve["frr"]], color="#3b82f6", label="FRR")
    axes[1].scatter([metrics["eer_threshold"]], [metrics["eer_percentage"]], color="black", zorder=5, label="EER")
    axes[1].set_title("FAR and FRR")
    axes[1].legend()
    axes[2].plot([v * 100 for v in curve["far"]], [v * 100 for v in curve["gar"]], color="#8b5cf6", label="AUC {:.4f}".format(metrics["auc"]))
    axes[2].plot([0, 100], [0, 100], linestyle=":", color="gray")
    axes[2].set_title("ROC")
    axes[2].legend()
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    print("Saved", path)
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        pass
    plt.close(fig)

def select_queries(altered):
    pool = altered if DIFFICULTY == "all" else [item for item in altered if item["difficulty"] == DIFFICULTY]
    if not pool:
        pool = list(altered)
    rank_d = {"easy": 0, "medium": 1, "hard": 2}
    rank_a = {"CR": 0, "Obl": 1, "Zcut": 2, "Unknown": 3}
    pool = sorted(pool, key=lambda item: (rank_d.get(item["difficulty"], 9), rank_a.get(item["alt"], 9), item["path"]))
    picked, seen = [], set()
    for item in pool:
        key = (item["difficulty"], item["alt"])
        if key in seen:
            continue
        seen.add(key)
        picked.append(item)
        if MAX_GENUINE_PER_FINGER and len(picked) >= MAX_GENUINE_PER_FINGER:
            break
    return picked

def load_template(path):
    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None
    center = (image.shape[1] // 2, image.shape[0] // 2)
    return extract_binary_template(image, center=center)

def evaluate_cohort(cohort_name, cohort_sids):
    print("\n" + "=" * 70)
    print(cohort_name, "—", len(cohort_sids), "subjects")
    print("=" * 70)
    records = []
    started = time.time()
    allowed = set(cohort_sids)
    finger_items = [(key, slot) for key, slot in fingers.items() if slot["sid"] in allowed and slot["real"]]
    for index, (key, slot) in enumerate(finger_items, start=1):
        enrolled_bits = load_template(slot["real"])
        if enrolled_bits is None:
            continue
        queries = []
        for item in select_queries(slot["altered"]):
            query_bits = load_template(item["path"])
            if query_bits is not None:
                queries.append((item["alt"], query_bits))
        records.append({"sid": slot["sid"], "finger": key, "bits": enrolled_bits, "queries": queries})
        if index % 200 == 0 or index == len(finger_items):
            print("  templates {}/{} ({:.0f}s)".format(index, len(finger_items), time.time() - started))

    genuine, impostor = [], []
    genuine_ok = zkp_ok = impostor_ok = 0
    by_alt = defaultdict(lambda: {"trials": 0, "accepted": 0})
    rng = np.random.RandomState(SPLIT_SEED)
    by_sid = defaultdict(list)
    for record in records:
        by_sid[record["sid"]].append(record)

    for record in records:
        helper, commit, public, _secret = fuzzy_commit(record["bits"])
        for alt_name, query_bits in record["queries"]:
            distance = best_hamming(record["bits"], query_bits)
            accepted, zkp_passed = crypto_decision(query_bits, helper, commit, public)
            genuine.append(distance)
            genuine_ok += int(accepted)
            zkp_ok += int(zkp_passed)
            by_alt[alt_name]["trials"] += 1
            by_alt[alt_name]["accepted"] += int(accepted)
        other_sids = [sid for sid in by_sid if sid != record["sid"]]
        if not other_sids:
            continue
        chosen = rng.choice(other_sids, size=min(IMPOSTORS_PER_FINGER, len(other_sids)), replace=False)
        for sid in np.atleast_1d(chosen):
            attacker = by_sid[int(sid)][rng.randint(0, len(by_sid[int(sid)]))]
            distance = best_hamming(record["bits"], attacker["bits"])
            accepted, _zkp = crypto_decision(attacker["bits"], helper, commit, public)
            impostor.append(distance)
            impostor_ok += int(accepted)

    if not genuine:
        raise RuntimeError("No genuine trials. Confirm the dataset has an Altered folder next to Real.")
    if not impostor:
        raise RuntimeError(
            cohort_name + " needs at least two subjects so impostor trials exist. "
            "On the full SOCOFing set the 20% test split still has about 120 subjects."
        )

    metrics = roc_and_eer(genuine, impostor)
    far = impostor_ok / len(impostor)
    frr = 1.0 - genuine_ok / len(genuine)
    metrics.update({
        "cohort_name": cohort_name,
        "num_subjects": len(set(record["sid"] for record in records)),
        "num_fingers": len(records),
        "total_genuine_attempts": len(genuine),
        "genuine_crypto_success": genuine_ok,
        "genuine_zkp_success": zkp_ok,
        "total_impostor_attempts": len(impostor),
        "impostor_crypto_success": impostor_ok,
        "cryptographic_far_percentage": round(far * 100.0, 4),
        "cryptographic_frr_percentage": round(frr * 100.0, 4),
        "cryptographic_gar_percentage": round((1.0 - frr) * 100.0, 4),
        "alteration_breakdown": {
            name: {
                "trials": bucket["trials"],
                "accepted": bucket["accepted"],
                "frr_percentage": round((1.0 - bucket["accepted"] / bucket["trials"]) * 100.0, 4) if bucket["trials"] else None,
            }
            for name, bucket in sorted(by_alt.items())
        },
    })
    safe = cohort_name.lower().replace(" ", "_")
    plot_path = safe + "_summary.png"
    if "train" in safe:
        plot_path = "kaggle_train_summary.png"
    elif "test" in safe:
        plot_path = "kaggle_test_summary.png"
    save_summary_plot(np.asarray(genuine), np.asarray(impostor), metrics, cohort_name, plot_path)
    metrics["plot_file"] = plot_path
    with open(plot_path, "rb") as handle:
        metrics["plot_b64"] = base64.b64encode(handle.read()).decode("ascii")

    print("Subjects / fingers:        {} / {}".format(metrics["num_subjects"], metrics["num_fingers"]))
    print("Genuine trials:            {}".format(len(genuine)))
    print("Genuine crypto accepted:   {} ({:.2f}%)".format(genuine_ok, (1.0 - frr) * 100.0))
    print("Genuine ZKP verified:      {}".format(zkp_ok))
    print("Impostor trials:           {}".format(len(impostor)))
    print("Impostor breaches:         {}".format(impostor_ok))
    print("Cryptographic FAR:         {}%".format(metrics["cryptographic_far_percentage"]))
    print("Cryptographic FRR:         {}%".format(metrics["cryptographic_frr_percentage"]))
    print("EER:                       {}% at {}".format(metrics["eer_percentage"], metrics["eer_threshold"]))
    print("AUC:                       {}".format(metrics["auc"]))
    print("Mean genuine HD:           {} +/- {}".format(metrics["mean_genuine_hd"], metrics["std_genuine_hd"]))
    print("Mean impostor HD:          {} +/- {}".format(metrics["mean_impostor_hd"], metrics["std_impostor_hd"]))
    print("Per alteration:", json.dumps(metrics["alteration_breakdown"]))
    return metrics


In [ ]:
rng = np.random.RandomState(SPLIT_SEED)
order = np.array(subjects)
rng.shuffle(order)
cut = int(round(len(order) * TRAIN_FRACTION))
cut = min(max(cut, 1), len(order) - 1) if len(order) > 1 else len(order)
train_subjects = sorted(int(v) for v in order[:cut])
test_subjects = sorted(int(v) for v in order[cut:])
print("Train subjects:", len(train_subjects), "| Test subjects:", len(test_subjects))
print("Difficulty filter:", DIFFICULTY, "| genuine cap per finger:", MAX_GENUINE_PER_FINGER)

train_results = evaluate_cohort("Kaggle SOCOFing Train", train_subjects)
test_results = evaluate_cohort("Kaggle SOCOFing Test", test_subjects) if test_subjects else None


In [ ]:
def public_view(metrics):
    hidden = {"plot_b64"}
    return {key: value for key, value in metrics.items() if key not in hidden}

export = {
    "dataset": "SOCOFing",
    "protocol": "Subject-disjoint {:.0f}/{:.0f} split, seed {}. Genuine = same finger, real vs altered. Impostor = different subject.".format(
        TRAIN_FRACTION * 100, (1 - TRAIN_FRACTION) * 100, SPLIT_SEED),
    "difficulty": DIFFICULTY,
    "kaggle_train": train_results,
    "kaggle_test": test_results,
}
with open("evaluation_results.json", "w") as handle:
    json.dump(export, handle)
print("Wrote evaluation_results.json")
print("\nTRAIN")
print(json.dumps(public_view(train_results), indent=2)[:2500])
if test_results:
    print("\nTEST")
    print(json.dumps(public_view(test_results), indent=2)[:2500])
print("\nDownload evaluation_results.json, kaggle_train_summary.png, and kaggle_test_summary.png.")
print("Put the JSON in the project data/ folder and the PNGs in evaluation/plots/, then start the website.")
